# Préparation de la table SYGEN - SAE 2025

## Objectif

Préparer une table **au niveau établissement** contenant les informations de SYGEN utiles au projet sur les inégalités territoriales d'accès aux soins :

- capacités hospitalières ;
- présence de certaines activités/autorisations ;
- personnel ;
- équipements lourds.

La table obtenue sera ensuite fusionnée avec **FINESS** grâce à l'identifiant `FI` afin de récupérer le type d'établissement et sa localisation communale.

**Source :** Base administrative SAE 2025 - DREES. La documentation décrit SYGEN comme la fiche de synthèse « Activité générale et activités autorisées ».

> Choix méthodologique : on conserve ici surtout des variables d'**offre/capacité**. Les volumes d'activité (séjours, journées, etc.) ne sont pas retenus dans la table finale, car la documentation SAE avertit que certaines données d'activité pré-remplies depuis le PMSI ne doivent pas être utilisées pour suivre l'activité des établissements.

In [1]:
import os
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)

In [2]:
path_sygen = "https://minio.lab.sspcloud.fr/mangane/Projet_PDS_2026/raw/SYGEN_2025.csv"

In [3]:
df_raw = pd.read_csv(
    path_sygen,
    sep=";",
    decimal=",",
    dtype={"FI": "string", "FI_EJ": "string", "AN": "Int64"}
)

print("Dimensions :", df_raw.shape)
print("Nombre de variables :", df_raw.shape[1])
df_raw.head()

Dimensions : (3988, 236)
Nombre de variables : 236


,FI,FI_EJ,AN,LIT_MED,LIT_CHI,LIT_OBS,LIT_MCO,SEJHC_MED,SEJHC_CHI,SEJHC_OBS,SEJHC_MCO,JOU_MED,JOU_CHI,JOU_OBS,JOU_MCO,PLA_MED,PLA_CHI,PLA_OBS,PLA_MCO,SEJHP_MED,SEJHP_CHI,SEJHP_OBS,SEJHP_MCO,SEANCES_MED,SPED,...,PHARMA_A26,NB_SCAN,NB_IRM,NB_MAMM,NB_LITHO,NB_RADIO,NB_RADIOINT,ACT_SCAN,ACT_IRM,ACT_MAMM,ACT_LITHO,SAL_INTERV,SALTOT24,SALAMBU,ACT_BLOCS,SURV_TOT,POST_REVEIL,NB_GAMMACAM,NB_TEP,NB_TEPSCAN,NB_TEPIRM,ACT_GAMMACAM,ACT_TEP,ACT_TEPSCAN,ACT_TEPIRM
0,010000024,010780054,2025,240.0,68.0,36.0,344.0,11486.0,3897.0,3365.0,18748.0,81253.0,18648.0,10135.0,110036.0,22.0,20.0,4.0,46.0,4754.0,4005.0,655.0,9414.0,40639.0,1.0,...,NaN,2.0,3.0,0.0,0.0,4.0,0.0,16190.0,9662.0,0.0,0.0,12.0,NaN,NaN,16021.0,2.0,20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,010000032,010780062,2025,56.0,19.0,8.0,83.0,3627.0,948.0,409.0,4984.0,19249.0,4019.0,1424.0,24692.0,8.0,8.0,1.0,17.0,156.0,1961.0,47.0,2164.0,752.0,1.0,...,NaN,1.0,0.0,1.0,0.0,2.0,0.0,9172.0,0.0,290.0,0.0,4.0,3.0,0.0,4185.0,1.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,010000065,010780096,2025,59.0,0.0,0.0,59.0,2165.0,0.0,0.0,2165.0,19825.0,0.0,0.0,19825.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,NaN,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,010000081,010780112,2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,010000099,010780120,2025,10.0,NaN,NaN,10.0,166.0,0.0,0.0,166.0,2070.0,0.0,0.0,2070.0,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
print("Nombre de lignes :", len(df_raw))
print("FI uniques :", df_raw["FI"].nunique())
print("Doublons FI :", df_raw["FI"].duplicated().sum())
print("FI manquants :", df_raw["FI"].isna().sum())
print("Années présentes :", sorted(df_raw["AN"].dropna().unique().tolist()))

Nombre de lignes : 3988
FI uniques : 3988
Doublons FI : 0
FI manquants : 0
Années présentes : [2025]


La sélection suivante vise l'**offre de soins** plutôt que les volumes d'activité.

- **MCO** : lits et places en médecine, chirurgie, obstétrique et total MCO ;
- **Psychiatrie** : capacité d'hospitalisation complète et hôpital de jour (totaux) ;
- **SMR / USLD / HAD** : capacités spécifiques ;
- **Services** : autorisations d'urgences générales/pédiatriques, SAMU, réanimation ;
- **Personnel** : effectifs/ETP médicaux et personnel non médical ;
- **Équipements** : scanner, IRM, mammographe.

In [7]:
colonnes_sygen = [
    # Identifiants
    "FI", "FI_EJ", "AN",

    # Capacités MCO
    "LIT_MED", "LIT_CHI", "LIT_OBS", "LIT_MCO",
    "PLA_MED", "PLA_CHI", "PLA_OBS", "PLA_MCO",

    # Psychiatrie
    "CAP_HC_TOT", "CAP_HDJ_TOT",

    # SMR, USLD et HAD
    "LIT_SMR", "PLA_SMR", "LIT_USLD", "PLATOT_HAD",

    # Présence / autorisation de services
    "AUTGEN", "AUTPED", "SAMU", "REA",

    # Personnel
    "EFFSAL_TOT", "EFFLIB_TOT", "ETPSAL_TOT",
    "EFFPNM_TOT_PNM", "ETPPNM_TOT_PNM",

    # Équipements
    "NB_SCAN", "NB_IRM", "NB_MAMM",
]


df_sygen = df_raw[colonnes_sygen].copy()
print("Dimensions de la table :", df_sygen.shape)
df_sygen.head()

Dimensions de la table : (3988, 29)


,FI,FI_EJ,AN,LIT_MED,LIT_CHI,LIT_OBS,LIT_MCO,PLA_MED,PLA_CHI,PLA_OBS,PLA_MCO,CAP_HC_TOT,CAP_HDJ_TOT,LIT_SMR,PLA_SMR,LIT_USLD,PLATOT_HAD,AUTGEN,AUTPED,SAMU,REA,EFFSAL_TOT,EFFLIB_TOT,ETPSAL_TOT,EFFPNM_TOT_PNM,ETPPNM_TOT_PNM,NB_SCAN,NB_IRM,NB_MAMM
0,010000024,010780054,2025,240.0,68.0,36.0,344.0,22.0,20.0,4.0,46.0,NaN,NaN,42.0,17.0,60.0,50.0,1.0,0.0,1.0,1.0,224.0,8.0,181.07,1952.0,1811.92,2.0,3.0,0.0
1,010000032,010780062,2025,56.0,19.0,8.0,83.0,8.0,8.0,1.0,17.0,NaN,NaN,40.0,8.0,NaN,NaN,1.0,0.0,0.0,0.0,45.0,0.0,27.69,438.0,380.05,1.0,0.0,1.0
2,010000065,010780096,2025,59.0,0.0,0.0,59.0,0.0,0.0,0.0,0.0,NaN,NaN,73.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,24.0,0.0,18.26,264.0,225.23,0.0,0.0,0.0
3,010000081,010780112,2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.0,0.0,30.0,NaN,NaN,NaN,NaN,NaN,2.0,0.0,1.83,77.0,64.16,NaN,NaN,NaN
4,010000099,010780120,2025,10.0,NaN,NaN,10.0,NaN,NaN,NaN,NaN,NaN,NaN,20.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,4.0,0.0,2.59,66.0,42.99,NaN,NaN,NaN


In [6]:
resume_qualite = pd.DataFrame({
    "nb_manquants": df_sygen.isna().sum(),
    "pct_manquants": (df_sygen.isna().mean() * 100).round(1),
})

colonnes_num = [c for c in df_sygen.select_dtypes(include="number").columns if c != "AN"]

resume_qualite["nb_zeros"] = [
    (df_sygen[c] == 0).sum() if c in colonnes_num else pd.NA
    for c in resume_qualite.index
]
resume_qualite["nb_positifs"] = [
    (df_sygen[c] > 0).sum() if c in colonnes_num else pd.NA
    for c in resume_qualite.index
]

resume_qualite.sort_values("pct_manquants", ascending=False)

,nb_manquants,pct_manquants,nb_zeros,nb_positifs
PLATOT_HAD,3708,93.0,0,280
SAMU,3530,88.5,357,101
CAP_HDJ_TOT,3442,86.3,15,531
PLA_OBS,3440,86.3,223,325
LIT_USLD,3416,85.7,1,571
AUTGEN,3371,84.5,16,601
AUTPED,3371,84.5,528,89
CAP_HC_TOT,3354,84.1,113,521
LIT_OBS,3328,83.5,211,449
PLA_CHI,3031,76.0,81,876


In [8]:
variables_capacite = [
    "LIT_MCO", "PLA_MCO", "CAP_HC_TOT", "CAP_HDJ_TOT",
    "LIT_SMR", "PLA_SMR", "LIT_USLD", "PLATOT_HAD",
    "NB_SCAN", "NB_IRM", "NB_MAMM"
]

df_sygen[variables_capacite].describe().T[["count", "mean", "50%", "max"]].rename(
    columns={"50%": "mediane"}
)

,count,mean,mediane,max
LIT_MCO,1444.0,128.069945,60.5,1275.0
PLA_MCO,1281.0,33.030445,24.0,249.0
CAP_HC_TOT,634.0,88.356467,63.5,1217.0
CAP_HDJ_TOT,546.0,54.014652,30.0,428.0
LIT_SMR,1785.0,56.612325,45.0,340.0
PLA_SMR,1785.0,12.146779,5.0,210.0
LIT_USLD,572.0,50.354895,40.0,305.0
PLATOT_HAD,280.0,95.907143,60.0,2048.0
NB_SCAN,1095.0,0.863014,1.0,7.0
NB_IRM,1095.0,0.714155,0.0,8.0


À ce stade, aucune imputation n'est effectuée et aucune ligne n'est supprimée :

- `FI` est la clé de fusion avec FINESS 
- les `NaN` sont conservés 
- les variables d'activité (séjours, journées, séances...) ne sont pas utilisées dans la table finale 
- la localisation communale sera ajoutée avec FINESS

In [13]:
df_sygen_clean = df_sygen.copy()

print("Dimensions finales :", df_sygen_clean.shape)
print("Doublons FI :", df_sygen_clean["FI"].duplicated().sum())
print("FI manquants :", df_sygen_clean["FI"].isna().sum())

df_sygen_clean.head()

Dimensions finales : (3988, 29)
Doublons FI : 0
FI manquants : 0


,FI,FI_EJ,AN,LIT_MED,LIT_CHI,LIT_OBS,LIT_MCO,PLA_MED,PLA_CHI,PLA_OBS,PLA_MCO,CAP_HC_TOT,CAP_HDJ_TOT,LIT_SMR,PLA_SMR,LIT_USLD,PLATOT_HAD,AUTGEN,AUTPED,SAMU,REA,EFFSAL_TOT,EFFLIB_TOT,ETPSAL_TOT,EFFPNM_TOT_PNM,ETPPNM_TOT_PNM,NB_SCAN,NB_IRM,NB_MAMM
0,010000024,010780054,2025,240.0,68.0,36.0,344.0,22.0,20.0,4.0,46.0,NaN,NaN,42.0,17.0,60.0,50.0,1.0,0.0,1.0,1.0,224.0,8.0,181.07,1952.0,1811.92,2.0,3.0,0.0
1,010000032,010780062,2025,56.0,19.0,8.0,83.0,8.0,8.0,1.0,17.0,NaN,NaN,40.0,8.0,NaN,NaN,1.0,0.0,0.0,0.0,45.0,0.0,27.69,438.0,380.05,1.0,0.0,1.0
2,010000065,010780096,2025,59.0,0.0,0.0,59.0,0.0,0.0,0.0,0.0,NaN,NaN,73.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,24.0,0.0,18.26,264.0,225.23,0.0,0.0,0.0
3,010000081,010780112,2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.0,0.0,30.0,NaN,NaN,NaN,NaN,NaN,2.0,0.0,1.83,77.0,64.16,NaN,NaN,NaN
4,010000099,010780120,2025,10.0,NaN,NaN,10.0,NaN,NaN,NaN,NaN,NaN,NaN,20.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,4.0,0.0,2.59,66.0,42.99,NaN,NaN,NaN


In [15]:
output_path = "s3://mangane/Projet_PDS_2026/processed/sygen_2025_clean.csv"

df_sygen_clean.to_csv(
    output_path,
    index=False
)

PermissionError: The Access Key Id you provided does not exist in our records.